In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
from flax import nnx

import ai_surrogate_models as ai
from scattering_simulation import prepare_lens_scattering_solver
import topology_parametrization
import field_postprocessing
from manufacturing_constraints import too_thin_area

import optax
from mutax import differential_evolution

from field_plotter import visible_amplitude_colormap
import matplotlib.pyplot as plt

In [ ]:
jax.config.update("jax_default_matmul_precision", "float32")
print(jax.devices())

## Lens geometry and constraints

In [ ]:
period = 2000
p_red = period / 650
p_green = period / 550
p_blue = period / 450
focal_distance = 3500

In [ ]:
n_px = 100
topology = topology_parametrization.GaussianField(n_px, sigma=16, symmetry_type='main_diagonal')

In [ ]:
# TODO: maybe keep n_px the same, just increase it to 200. This would increase the size of x but FNO stays the same
# TODO: check if this affects the time significantly on the server
# TODO: only this part of code changes
n_px_constraints = 200
min_width = 150
min_width_px = round(min_width * n_px_constraints / period)
min_width_px = (min_width_px // 2) * 2 + 1
print('Min width in pixels:', min_width_px, 'which corresponds to', min_width_px * period / n_px_constraints, 'nm')

## Load FNO models

In [ ]:
common_fno_kwargs = {'model_type': ai.ScatteringFNO, 'dout': 4, 'hidden_dims': [64] * 7, 'modes': 16, 'activation': nnx.gelu}
red_model = ai.load_model('ai_models/fno_red_freeform.msgpack', **common_fno_kwargs)
green_model = ai.load_model('ai_models/fno_green_freeform.msgpack', **common_fno_kwargs)
blue_model = ai.load_model('ai_models/fno_blue_freeform.msgpack', **common_fno_kwargs)

## Prepare rigorous simulation functions

In [ ]:
common_sim_kwargs = {'period': 2000, 'lens_thickness': 600, 'substrate_thickness': 500,
                     'approximate_number_of_terms': 600, 'propagate_by_distance': 0}
red_sim, red_basis = prepare_lens_scattering_solver(wavelength=650, **common_sim_kwargs)
green_sim, green_basis = prepare_lens_scattering_solver(wavelength=550, **common_sim_kwargs)
blue_sim, blue_basis = prepare_lens_scattering_solver(wavelength=450, **common_sim_kwargs)

print(red_basis.shape, green_basis.shape, blue_basis.shape)

## Efficiency and objective

In [ ]:
def params_to_normalized_amps_xy_rgb(params, sim_func_red, sim_func_green, sim_func_blue):
    amps_red = sim_func_red(params).squeeze()[:red_basis.shape[1]].T
    amps_green = sim_func_green(params).squeeze()[:green_basis.shape[1]].T
    amps_blue = sim_func_blue(params).squeeze()[:blue_basis.shape[1]].T

    amps_red = field_postprocessing.normalize_power(amps_red, p_red, red_basis)[:2]
    amps_green = field_postprocessing.normalize_power(amps_green, p_green, green_basis)[:2]
    amps_blue = field_postprocessing.normalize_power(amps_blue, p_blue, blue_basis)[:2]

    amps_red = field_postprocessing.propagate_amps_in_free_space(amps_red, focal_distance, red_basis, 650, period)
    amps_green = field_postprocessing.propagate_amps_in_free_space(amps_green, focal_distance, green_basis, 550, period)
    amps_blue = field_postprocessing.propagate_amps_in_free_space(amps_blue, focal_distance, blue_basis, 450, period)

    return amps_red, amps_green, amps_blue

def params_to_focused_powers(params, true_sim=True):
    if true_sim:
        sim_funcs = (
            lambda x: red_sim(topology(x)),
            lambda x: green_sim(topology(x)),
            lambda x: blue_sim(topology(x))
        )
    else:
        sim_funcs = (
            lambda x: field_postprocessing.extract_amps_from_fields(red_model(topology(x)[None])[0], red_basis),
            lambda x: field_postprocessing.extract_amps_from_fields(green_model(topology(x)[None])[0], green_basis),
            lambda x: field_postprocessing.extract_amps_from_fields(blue_model(topology(x)[None])[0], blue_basis)
        )

    amps_red, amps_green, amps_blue = params_to_normalized_amps_xy_rgb(params, *sim_funcs)

    red_power = field_postprocessing.calculate_focused_power(amps_red, p_red, red_basis, relative_focal_point=(0.25, 0.25))
    green_power = (
        field_postprocessing.calculate_focused_power(amps_green, p_green, green_basis, relative_focal_point=(0.25, 0.75))
        + field_postprocessing.calculate_focused_power(amps_green, p_green, green_basis, relative_focal_point=(0.75, 0.25))
    )
    blue_power = field_postprocessing.calculate_focused_power(amps_blue, p_blue, blue_basis, relative_focal_point=(0.75, 0.75))

    return jnp.array([red_power, green_power, blue_power])

In [ ]:
def objective(x, true_sim, penalty_coeff):
    eff = jnp.mean(params_to_focused_powers(x, true_sim=true_sim))
    invalid = too_thin_area(topology(x, n_px_constraints), min_width_px)
    return eff - (jnp.exp(penalty_coeff * invalid) - 1)

## Random restart gragient descent FNO-based optimization

In [ ]:
learning_rate = 1e-2
loss_value_and_grad_fn = jax.value_and_grad(lambda x, pc: objective(x, true_sim=False, penalty_coeff=pc), argnums=0)
optimizer = optax.adam(learning_rate)

n_restarts = 5
n_steps = 500

# penalty_coeff_schedule = optax.schedules.exponential_decay(init_value=1, decay_rate=100, transition_steps=n_steps)
penalty_coeff_schedule = optax.schedules.linear_schedule(init_value=1, end_value=100, transition_steps=int(n_steps * 0.8))

overall_best_x = None
overall_max_f = -1e10
max_f_history = []
final_x_history = []

@jax.jit
def ai_manygrad_step(x, opt_state, penalty_coeff):
    obj_f, grad = loss_value_and_grad_fn(x, penalty_coeff)
    updates, opt_state = optimizer.update(-grad, opt_state)
    x = optax.apply_updates(x, updates)
    x = jnp.clip(x, topology.minval, topology.maxval)
    return x, opt_state, obj_f, grad

for rng_seed in range(n_restarts):
    x_init = jax.random.uniform(
        jax.random.key(rng_seed),
        shape=topology.n_geometrical_parameters,
        minval=topology.minval,
        maxval=topology.maxval
    )
    opt_state = optimizer.init(x_init)

    x = x_init
    max_f = -1e10
    best_x = x

    for i in range(n_steps):
        new_x, opt_state, obj_f, grad = ai_manygrad_step(x, opt_state, penalty_coeff_schedule(i))
        if obj_f > max_f:
            max_f = obj_f
            best_x = x
        x = new_x
        print(i, obj_f, sep='\t')

    if max_f > overall_max_f:
        overall_max_f = max_f
        overall_best_x = best_x
    max_f_history.append(max_f)
    final_x_history.append(x)
    print(rng_seed, max_f, overall_max_f)

TODO: count best only among the final 100 (where scheduling ended)

In [ ]:
x_to_inspect = final_x_history[4]
print(max_f_history[4])
plt.imshow(topology(x_to_inspect, 220))

In [ ]:
too_thin_area(topology(x_to_inspect, n_px_constraints), min_width_px)

In [ ]:
np.exp(100 * too_thin_area(topology(x_to_inspect, n_px_constraints), min_width_px)) - 1

In [ ]:
from manufacturing_constraints import detect_too_thin_solid, detect_too_thin_void

In [ ]:
n_px_1 = 200
min_width_px_1 = round(min_width * n_px_1 / period)
min_width_px_1 = (min_width_px_1 // 2) * 2 + 1
print('Min width in pixels:', min_width_px_1, 'which corresponds to', min_width_px_1 * period / n_px_1, 'nm')

In [ ]:
pattern = topology(x_to_inspect, n_px_1)
invalid_solid = detect_too_thin_solid(pattern, min_width_px_1)
plt.imshow(np.stack([pattern, pattern - invalid_solid, pattern - invalid_solid], axis=-1))
plt.axis('off')
plt.show()

In [ ]:
invalid_void = detect_too_thin_void(pattern, min_width_px_1)
plt.imshow(np.stack([invalid_void + pattern, pattern, pattern], axis=-1))
plt.axis('off')
plt.show()

# Choose and save TOP candidates

In [ ]:
n_top = 10

# Rigorous Refinement

Тут раз я всё равно переделываю с нуля всё можно провести на сервере сравнение между FNO и FMM во время FNO оптимизации только для тех сидов, которые были TOP 5-10 в предоптимизации. И дальше получится красивая картинка где линия FMM дальше продолжается в Refinement. Надо аккуратно это оформить тут перед тем как на сервере запускать

Симуляцию с углами наклона и расширением границ можно не делать на сервере, это отдельно уже можно локально

Надо сохранить параметры (не геометрии) всех линз, оптимизированных FMM из TOP 10. Тогда можно будет их сравнить по устойчивости к наклону и ошибкам изготовления

Broadband вообще не смотрю, это отдельная огромная работа. Пусть как и было всё работает плохо

Надо наверное ставить размер 150 чтобы не было проблем ну и результат солиднее если будет хорошая эффективность